In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gould_2026.datasets import LDS
from gould_2026.stim_designer import StimDesigner, OptimizationMethod
from tqdm.autonotebook import tqdm

rng = np.random.default_rng(1)


In [ ]:

stim_scale = .5
def sim(v, optimization_method,rng):
    sd = StimDesigner(rng_seed=rng.integers(0,2**32), optimization_method=optimization_method)


    lds = LDS.circular_lds(obs_d=2, process_noise=1e-5, obs_noise=1e-10, obs_center=np.array([0,0]), rng=rng)
    lds.B = np.eye(2)
    lds.C = np.eye(2)
    LDS.A = lds.A
    lds.has_been_stimmed = False

    s_designed = None

    def U(lds, state, i, rng):
        nonlocal s_designed

        if i > 100 and not lds.has_been_stimmed and np.abs(np.atan2(state[1], state[0]) - np.pi/2) < np.pi/50:
            lds.has_been_stimmed = True

            u = sd.design_stim(v=v, u_to_s_function=lambda u: u, u_dimension=2, equivalent_projection_matrix=np.eye(2))
            if optimization_method == OptimizationMethod.CHEAT_LOWD_VEC:
                assert np.array_equal(u.flatten(), v.flatten())

            s_designed = u * stim_scale

            return s_designed
        else:
            return np.array([0, 0])

    states, observations, U = lds.simulate(200, initial_state=np.array([1,0]), U=U, rng=rng)

    stim_idx = np.argmax(np.linalg.norm(U, axis=1))

    estimated_A = np.linalg.lstsq(observations[:stim_idx-2], observations[1:stim_idx-1])[0]

    s_obs = observations[stim_idx] - observations[stim_idx-1] @ estimated_A


    return observations, U, lds, s_designed, s_obs, stim_idx, v

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(10,5), squeeze=False, sharex=True, sharey=True)

v = np.array([1,1])[:,None]
seed = rng.integers(0,2**32)
for ax_idx, optimization_method in enumerate([OptimizationMethod.JAXOPT, OptimizationMethod.CHEAT_LOWD_VEC]):
    observations, U, lds, s_designed, s_obs, stim_idx, _ = sim(v=v, optimization_method=optimization_method, rng=np.random.default_rng(seed))
    axs[0,ax_idx].plot(observations[:,0], observations[:,1])
    axs[0,ax_idx].plot(observations[stim_idx - 1, 0], observations[stim_idx - 1, 1], 'r.')
    axs[0,ax_idx].axis("scaled")
    axs[0,ax_idx].axis("off")

axs[0,0].set_title("Our optimization")
axs[0,1].set_title("Optimal stimulation")


In [ ]:
l = []

for _ in tqdm(range(500)):
    observations, U, lds, s_designed, s_obs, stim_idx, inner_v = sim(v=v, optimization_method=OptimizationMethod.JAXOPT, rng=np.random.default_rng())
    l.append(s_designed - stim_scale * inner_v.flatten())


In [ ]:
errors = np.linalg.norm(l, axis=1) / np.linalg.norm(inner_v * stim_scale)
plt.hist(errors, bins=100);

threshold = 0.025
plt.axvline(threshold, color='k', linestyle='--')
print((errors > threshold).sum())